# Phase 2 - Baseline Reproduction

This notebook reproduces the official baseline locally from the repository root. It uses only the official files in `data/input/`, generates the first baseline submission, and updates the experiment log.

This is not a model-improvement notebook. It does not tune hyperparameters, compare model families, use external data, or add features beyond the BMI feature explicitly present in the official baseline workflow.


## Baseline Contract

- Target column: `Drafted`
- ID column: `Id`
- Metric: ROC-AUC using positive-class probabilities
- Submission path: `outputs/submissions/submission_001_baseline.csv`
- Experiment log: `logs/experiment_log.csv`

Official baseline BMI feature: BMI is included only because it appears in the official baseline workflow. It is not introduced here as new Phase 2 feature engineering.


In [1]:
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder


In [2]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "input"
SUBMISSION_DIR = PROJECT_ROOT / "outputs" / "submissions"
SUBMISSION_PATH = SUBMISSION_DIR / "submission_001_baseline.csv"
LOG_PATH = PROJECT_ROOT / "logs" / "experiment_log.csv"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [TRAIN_PATH, TEST_PATH, SAMPLE_SUBMISSION_PATH, LOG_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, f"Missing required files: {missing_paths}"
assert DATA_DIR == PROJECT_ROOT / "data" / "input", "Data must be loaded only from data/input/."
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Submission path: {SUBMISSION_PATH}")


Project root: C:\GitHub\reto_Tokio
Data directory: C:\GitHub\reto_Tokio\data\input
Submission path: C:\GitHub\reto_Tokio\outputs\submissions\submission_001_baseline.csv


In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

TARGET_COLUMN = "Drafted"
ID_COLUMN = "Id"
EXPECTED_TRAIN_SHAPE = (2781, 16)
EXPECTED_TEST_SHAPE = (696, 15)
EXPECTED_SAMPLE_SHAPE = (696, 2)
EXPECTED_SUBMISSION_COLUMNS = ["Id", "Drafted"]

assert train_raw.shape == EXPECTED_TRAIN_SHAPE, f"Unexpected train shape: {train_raw.shape}"
assert test_raw.shape == EXPECTED_TEST_SHAPE, f"Unexpected test shape: {test_raw.shape}"
assert sample_submission.shape == EXPECTED_SAMPLE_SHAPE, f"Unexpected sample submission shape: {sample_submission.shape}"
assert TARGET_COLUMN in train_raw.columns, "Drafted must exist in train."
assert TARGET_COLUMN not in test_raw.columns, "Drafted must not exist in test."
assert ID_COLUMN in train_raw.columns, "Id must exist in train."
assert ID_COLUMN in test_raw.columns, "Id must exist in test."
assert ID_COLUMN in sample_submission.columns, "Id must exist in sample submission."
assert list(sample_submission.columns) == EXPECTED_SUBMISSION_COLUMNS, sample_submission.columns.tolist()
assert list(test_raw.columns) == [col for col in train_raw.columns if col != TARGET_COLUMN], "Test columns must match train columns after removing target."
assert sample_submission[ID_COLUMN].equals(test_raw[ID_COLUMN]), "Sample submission IDs must match test IDs in order."
assert set(train_raw[TARGET_COLUMN].dropna().astype(int).unique()) == {0, 1}, "Target must be binary."

print(f"train: {train_raw.shape}")
print(f"test: {test_raw.shape}")
print(f"sample_submission: {sample_submission.shape}")
print("Pre-run contract checks passed.")


train: (2781, 16)
test: (696, 15)
sample_submission: (696, 2)
Pre-run contract checks passed.


## Official Baseline Preprocessing

The official baseline drops `Id` and `School`, imputes selected numeric columns using means computed from the training data, label-encodes the remaining categorical columns using encoders fitted on training data, and includes the official baseline BMI feature.


In [4]:
train_model = train_raw.copy()
test_model = test_raw.copy()

train_ids = train_model[ID_COLUMN].copy()
test_ids = test_model[ID_COLUMN].copy()

# Match the official baseline: drop Id and School before modeling.
train_model = train_model.drop(columns=[ID_COLUMN, "School"])
test_model = test_model.drop(columns=[ID_COLUMN, "School"])

cols_to_fill = [
    "Age",
    "Sprint_40yd",
    "Vertical_Jump",
    "Bench_Press_Reps",
    "Broad_Jump",
    "Agility_3cone",
    "Shuttle",
]

for col in cols_to_fill:
    mean_value = train_model[col].mean()
    train_model[col] = train_model[col].fillna(mean_value)
    test_model[col] = test_model[col].fillna(mean_value)

categorical_cols = ["Player_Type", "Position_Type", "Position"]
unknown_categories = {}
for col in categorical_cols:
    train_categories = set(train_model[col].astype(str))
    test_categories = set(test_model[col].astype(str))
    unseen = sorted(test_categories - train_categories)
    if unseen:
        unknown_categories[col] = unseen
assert not unknown_categories, f"Unseen test categories would break official LabelEncoder baseline: {unknown_categories}"

label_encoders = {}
for col in categorical_cols:
    encoder = LabelEncoder()
    train_model[col] = encoder.fit_transform(train_model[col].astype(str))
    test_model[col] = encoder.transform(test_model[col].astype(str))
    label_encoders[col] = encoder

# Official baseline BMI feature.
for df in [train_model, test_model]:
    df["BMI"] = df["Weight"] / (df["Height"] ** 2)

assert train_ids.equals(train_raw[ID_COLUMN])
assert test_ids.equals(test_raw[ID_COLUMN])
assert not train_model.isna().any().any(), "Train model matrix contains missing values."
assert not test_model.isna().any().any(), "Test model matrix contains missing values."

X = train_model.drop(columns=[TARGET_COLUMN])
y = train_model[TARGET_COLUMN].astype(int)
assert TARGET_COLUMN not in X.columns
assert list(test_model.columns) == list(X.columns), "Test feature columns must match training features."
print(f"Feature matrix: {X.shape}")
print(f"Test matrix: {test_model.shape}")
print("Official baseline preprocessing checks passed.")


Feature matrix: (2781, 14)
Test matrix: (696, 14)
Official baseline preprocessing checks passed.


## Official Baseline Validation

Validation uses the official baseline model and fold settings. Scores are computed with ROC-AUC from positive-class probabilities.


In [5]:
model_params = {
    "n_estimators": 100,
    "max_depth": 5,
    "random_state": 2025,
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_scores = []
test_pred_proba_list = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = RandomForestClassifier(**model_params)
    model.fit(X_train, y_train)

    y_valid_pred_proba = model.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, y_valid_pred_proba)
    fold_scores.append(auc)

    test_pred_proba = model.predict_proba(test_model)[:, 1]
    test_pred_proba_list.append(test_pred_proba)

    print(f"Fold {fold}: ROC-AUC = {auc:.6f}")

cv_auc_mean = float(np.mean(fold_scores))
cv_auc_std = float(np.std(fold_scores, ddof=0))
print(f"Mean ROC-AUC: {cv_auc_mean:.6f}")
print(f"Std ROC-AUC: {cv_auc_std:.6f}")


Fold 1: ROC-AUC = 0.786138


Fold 2: ROC-AUC = 0.835926


Fold 3: ROC-AUC = 0.837446


Fold 4: ROC-AUC = 0.777615


Fold 5: ROC-AUC = 0.827693
Mean ROC-AUC: 0.812964
Std ROC-AUC: 0.025740


## Generate and Validate Baseline Submission


In [6]:
test_pred_proba_mean = np.mean(test_pred_proba_list, axis=0)

submission = sample_submission.copy()
submission["Drafted"] = test_pred_proba_mean

assert list(submission.columns) == EXPECTED_SUBMISSION_COLUMNS, submission.columns.tolist()
assert len(submission) == len(sample_submission), "Submission row count must match sample submission."
assert submission[ID_COLUMN].equals(sample_submission[ID_COLUMN]), "Submission IDs must match sample submission order."
assert submission[ID_COLUMN].equals(test_raw[ID_COLUMN]), "Submission IDs must match test order."
assert pd.api.types.is_numeric_dtype(submission["Drafted"]), "Predictions must be numeric."
assert submission["Drafted"].notna().all(), "Predictions must not contain NaN."
assert np.isfinite(submission["Drafted"].to_numpy()).all(), "Predictions must be finite."
assert submission["Drafted"].between(0, 1).all(), "Predictions must be probabilities in [0, 1]."

submission.to_csv(SUBMISSION_PATH, index=False)

saved_submission = pd.read_csv(SUBMISSION_PATH)
assert saved_submission.shape == EXPECTED_SAMPLE_SHAPE, saved_submission.shape
assert list(saved_submission.columns) == EXPECTED_SUBMISSION_COLUMNS, saved_submission.columns.tolist()
assert saved_submission[ID_COLUMN].equals(sample_submission[ID_COLUMN]), "Saved submission IDs must match sample submission order."
assert saved_submission[ID_COLUMN].equals(test_raw[ID_COLUMN]), "Saved submission IDs must match test order."
assert saved_submission["Drafted"].notna().all(), "Saved predictions must not contain NaN."
assert np.isfinite(saved_submission["Drafted"].to_numpy()).all(), "Saved predictions must be finite."
assert saved_submission["Drafted"].between(0, 1).all(), "Saved predictions must be probabilities in [0, 1]."

prediction_min = float(saved_submission["Drafted"].min())
prediction_max = float(saved_submission["Drafted"].max())
print(f"Saved baseline submission: {SUBMISSION_PATH}")
print(f"Submission shape: {saved_submission.shape}")
print(f"Prediction range: [{prediction_min:.6f}, {prediction_max:.6f}]")
print("Submission checks passed.")


Saved baseline submission: C:\GitHub\reto_Tokio\outputs\submissions\submission_001_baseline.csv
Submission shape: (696, 2)
Prediction range: [0.153988, 0.910945]
Submission checks passed.


## Register Baseline Experiment

The log update is idempotent: rerunning this notebook replaces the existing `001_baseline_reproduction` row instead of appending duplicates.


In [7]:
expected_log_columns = [
    "experiment_id",
    "date",
    "phase",
    "notebook",
    "features_version",
    "model",
    "params_summary",
    "cv_auc_mean",
    "cv_auc_std",
    "leaderboard_auc",
    "submission_file",
    "risk_flags",
    "notes",
]

if LOG_PATH.exists() and LOG_PATH.stat().st_size > 0:
    experiment_log = pd.read_csv(LOG_PATH)
else:
    experiment_log = pd.DataFrame(columns=expected_log_columns)

assert list(experiment_log.columns) == expected_log_columns, experiment_log.columns.tolist()

experiment_id = "001_baseline_reproduction"
row = {
    "experiment_id": experiment_id,
    "date": date.today().isoformat(),
    "phase": "Phase 2 - Baseline Reproduction",
    "notebook": "notebooks/01_baseline_reproduction.ipynb",
    "features_version": "official_baseline_bmi",
    "model": "RandomForestClassifier",
    "params_summary": "n_estimators=100; max_depth=5; random_state=2025; cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    "cv_auc_mean": f"{cv_auc_mean:.6f}",
    "cv_auc_std": f"{cv_auc_std:.6f}",
    "leaderboard_auc": "not_submitted_yet",
    "submission_file": "outputs/submissions/submission_001_baseline.csv",
    "risk_flags": "official_baseline_reproduction; train_mean_imputation; label_encoding; no_external_data; not_leaderboard_tuned",
    "notes": "Reproduces official baseline locally with data/input paths and validated submission format.",
}

experiment_log = experiment_log[experiment_log["experiment_id"] != experiment_id]
experiment_log = pd.concat([experiment_log, pd.DataFrame([row])], ignore_index=True)
experiment_log = experiment_log[expected_log_columns]
experiment_log.to_csv(LOG_PATH, index=False)

print(f"Updated experiment log: {LOG_PATH}")
print(experiment_log[experiment_log["experiment_id"] == experiment_id].to_string(index=False))


Updated experiment log: C:\GitHub\reto_Tokio\logs\experiment_log.csv
            experiment_id       date                           phase                                 notebook      features_version                  model                                                                                                  params_summary cv_auc_mean cv_auc_std   leaderboard_auc                                 submission_file                                                                                                     risk_flags                                                                                       notes
001_baseline_reproduction 2026-06-10 Phase 2 - Baseline Reproduction notebooks/01_baseline_reproduction.ipynb official_baseline_bmi RandomForestClassifier n_estimators=100; max_depth=5; random_state=2025; cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)    0.812964   0.025740 not_submitted_yet outputs/submissions/submission_001_baseline.csv official_baselin